# IM19 Universal MCP — Google Colab CPU Runtime

Notebook untuk menjalankan repository StrategyG/im19-universal-mcp pada Colab CPU. Mode default **local development**: server MCP berjalan lewat subprocess stdio, memakai SQLite dan kredensial sintetis di runtime yang sementara.

**Tidak membuka port publik, tidak menyimpan kunci di GitHub, dan tidak melakukan Google OAuth otomatis.** Colab bukan server 24/7. Connector yang benar-benar tersedia di revisi ini adalah YouTube read-only dan Google Drive metadata/teks terbatas; file PDF/EPUB dan tool write belum tersedia.

Sumber kode dipin ke commit e60d8276f6cb0b87b30fb04d0819f788dcad2544 di branch feat/distributed-quotas-v1 (PR #5). Ini bukan klaim production-ready. Jalankan sel dari atas ke bawah.

In [ ]:
import sys, platform
assert sys.version_info >= (3, 11), "Python minimal 3.11 diperlukan"
print("Python", platform.python_version())
print("CPU runtime; GPU tidak diperlukan.")

## 1. Clone repository dan verifikasi SHA

Clone public GitHub, tidak memerlukan token. Jika folder clone sudah berisi perubahan lokal, proses dihentikan tanpa menimpanya.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/StrategyG/im19-universal-mcp.git"
REPO_BRANCH = "feat/distributed-quotas-v1"
PINNED_SHA = "e60d8276f6cb0b87b30fb04d0819f788dcad2544"
BASE_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = BASE_DIR / "im19-universal-mcp"

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError("Folder target sudah ada dan bukan clone Git.")
    subprocess.run(["git", "clone", "--single-branch", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if not remote.rstrip("/").endswith("StrategyG/im19-universal-mcp.git"):
        raise RuntimeError("Remote Git bukan repository IM19.")

dirty = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
).strip()
if dirty:
    raise RuntimeError("Clone memiliki perubahan lokal; tidak akan di-overwrite.")
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", PINNED_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert actual_sha == PINNED_SHA
print("Repository:", REPO_DIR)
print("SHA valid:", actual_sha)

## 2. Instal package dan verifikasi import di kernel

Notebook memasang package ke interpreter yang sama dengan kernel, menambahkan `src` ke `sys.path`, dan langsung memeriksa import `im19_mcp`. Cara ini menghindari kesalahan `ModuleNotFoundError` akibat path/cache Colab.


In [ ]:
# Pakai interpreter yang sama persis dengan kernel Colab. Tidak memakai !pip.
import importlib
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
REPO_SRC = (REPO_DIR / "src").resolve()
PACKAGE_DIR = REPO_SRC / "im19_mcp"
if not (PACKAGE_DIR / "readiness.py").is_file():
    raise FileNotFoundError(
        f"Modul IM19 tidak ditemukan: {PACKAGE_DIR}. "
        "Jalankan ulang sel clone repository dan cek nama repository."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]"],
    cwd=str(REPO_DIR),
    check=True,
)
# Colab dapat mempertahankan cache import dari sel sebelumnya; editable
# install kadang baru terbaca setelah kernel di-restart. Tambahkan src
# secara eksplisit supaya tidak perlu restart notebook.
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))
importlib.invalidate_caches()

import im19_mcp
from im19_mcp.readiness import REQUIRED_SCHEMA_REVISION
import mcp, sqlalchemy, alembic, cryptography
assert Path(im19_mcp.__file__).resolve().is_relative_to(PACKAGE_DIR)
print("IM19 IMPORT: SUCCESS", im19_mcp.__file__)
print("Expected database revision:", REQUIRED_SCHEMA_REVISION)
print("SQLAlchemy:", sqlalchemy.__version__)

## 3. Runtime dev tanpa akun Google

Kunci Fernet dan audit salt dibuat secara acak di RAM. Database SQLite di direktori sementara Colab. Tidak mengakses Google Drive Anda. Identitas dev lokal bersifat sintetis, bukan autentikasi sesungguhnya.

In [ ]:
import os, secrets, tempfile
from pathlib import Path
from cryptography.fernet import Fernet

RUNTIME_DIR = Path(tempfile.mkdtemp(
    prefix="im19-local-", dir="/content" if Path("/content").is_dir() else None
)).resolve()
LOCAL_SETTINGS = {
    "IM19_DEPLOYMENT_ENV": "development",
    "IM19_DATABASE_URL": "sqlite:///" + str(RUNTIME_DIR / "im19.sqlite3"),
    "IM19_FERNET_KEY": Fernet.generate_key().decode(),
    "IM19_AUDIT_SALT": secrets.token_hex(32),
    "IM19_DEV_SUBJECT": "colab-synthetic-test-user",
    "IM19_GOOGLE_CLIENT_ID": "",
    "IM19_GOOGLE_CLIENT_SECRET": "",
    "IM19_GOOGLE_REDIRECT_URI": "",
    "IM19_FERNET_PREVIOUS_KEYS": "",
}
# Minimal inherited settings; do not pass unrelated API secrets.
ALLOWED_PROCESS_ENV = {
    "PATH", "HOME", "LANG", "LC_ALL", "LC_CTYPE", "TMPDIR",
    "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE",
    "VIRTUAL_ENV", "LD_LIBRARY_PATH",
}
LOCAL_ENV = {k: os.environ[k] for k in ALLOWED_PROCESS_ENV if k in os.environ}
LOCAL_ENV.update(LOCAL_SETTINGS)
# Child Python used by "alembic" and MCP stdio must resolve local src.
# Do not inherit an uncontrolled PYTHONPATH from the Colab kernel.
LOCAL_ENV["PYTHONPATH"] = str(REPO_SRC)
print("Ephemeral runtime:", RUNTIME_DIR)
print("Google API connection:", "NOT_CONFIGURED")
print("Subprocess source path:", LOCAL_ENV["PYTHONPATH"])

## 4. Migrasi Alembic (dengan pengecekan ulang import)

Sel migrasi memverifikasi keberadaan modul, memperbaiki path untuk kernel dan subprocess, dan menampilkan error yang jelas jika sel setup belum dijalankan.


In [ ]:
# Self-repair for execution out of order: "Run all" is still recommended.
import importlib
import subprocess
import sys
from pathlib import Path
from sqlalchemy import create_engine

REPO_DIR = Path(globals().get("REPO_DIR", "/content/im19-universal-mcp")).resolve()
REPO_SRC = (REPO_DIR / "src").resolve()
if not (REPO_SRC / "im19_mcp" / "readiness.py").is_file():
    raise RuntimeError(
        f"IM19 tidak ditemukan di {REPO_SRC}. Jalankan sel clone repository."
    )
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))
importlib.invalidate_caches()

# Ensure SQLAlchemy/Alembic and MCP dependencies are installed in this kernel.
try:
    from im19_mcp.readiness import REQUIRED_SCHEMA_REVISION, database_ready
except ModuleNotFoundError as exc:
    if exc.name != "im19_mcp":
        raise
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[dev]"],
        cwd=str(REPO_DIR), check=True
    )
    importlib.invalidate_caches()
    from im19_mcp.readiness import REQUIRED_SCHEMA_REVISION, database_ready

if "LOCAL_ENV" not in globals() or "LOCAL_SETTINGS" not in globals():
    raise RuntimeError("Sel runtime lokal belum dijalankan; jalankan sel 3 dahulu.")
LOCAL_ENV["PYTHONPATH"] = str(REPO_SRC)
subprocess.run(
    [sys.executable, "-m", "alembic", "upgrade", "head"],
    cwd=str(REPO_DIR), env=LOCAL_ENV, check=True
)
db_engine = create_engine(LOCAL_SETTINGS["IM19_DATABASE_URL"])
try:
    assert database_ready(db_engine), (
        f"Database tidak ready; revision harus {REQUIRED_SCHEMA_REVISION}"
    )
    print("DATABASE_READY:", REQUIRED_SCHEMA_REVISION)
finally:
    db_engine.dispose()

## 5. Jalankan subprocess MCP stdio sesungguhnya

MCP SDK akan menginisialisasi subprocess IM19, menemukan tools, membaca kapabilitas, dan memastikan Google Drive belum terhubung. Tidak ada koneksi publik.

In [ ]:
import asyncio, json
from mcp import Client
from mcp.client.stdio import StdioServerParameters

STDIO = StdioServerParameters(
    command=sys.executable,
    args=["-m", "im19_mcp.server"],
    env=LOCAL_ENV,
)

def parse_mcp(result):
    if result.is_error:
        raise RuntimeError("Tool MCP gagal.")
    if result.structured_content is not None:
        return result.structured_content
    return json.loads(result.content[0].text)

async def smoke_test():
    async with Client(STDIO) as client:
        names = sorted(t.name for t in (await client.list_tools()).tools)
        required = {"list_available_services", "get_service_capabilities",
                    "get_connection_status", "drive_search_files",
                    "youtube_get_channel"}
        assert required.issubset(names), "Tool MCP tidak lengkap"
        catalog = parse_mcp(await client.call_tool("list_available_services", {}))
        drive = parse_mcp(await client.call_tool(
            "get_connection_status", {"service": "drive"}))
        assert catalog["live_connection_claimed"] is False
        assert drive["status"] == "AUTH_REQUIRED"
        return {"transport": "stdio", "tools": names,
                "drive_status": drive["status"], "google_live": False}

MCP_RESULT = await asyncio.wait_for(smoke_test(), timeout=45)
print(json.dumps(MCP_RESULT, indent=2, ensure_ascii=False))

## 6. Panggil tools diagnostik

Hanya tool tanpa token atau data pribadi yang diizinkan di demo ini. Jangan memasukkan API key, kode otorisasi, atau refresh token sebagai argumen MCP.

In [ ]:
TOOL_NAME = "get_service_capabilities"
TOOL_ARGS = {"service": "drive"}
SAFE_TOOLS = {"list_available_services", "get_service_capabilities",
              "get_connection_status"}
assert TOOL_NAME in SAFE_TOOLS

async def call_safe_tool(name, arguments):
    async with Client(STDIO) as client:
        return parse_mcp(await client.call_tool(name, arguments))

value = await asyncio.wait_for(call_safe_tool(TOOL_NAME, TOOL_ARGS), timeout=30)
print(json.dumps(value, indent=2, ensure_ascii=False))

## 7. Jalankan pengujian repository (opsional)

Tes mock/unit tidak menggunakan kredensial Google. Tes yang membutuhkan PostgreSQL CI akan dilewati. Status tetap bergantung pada environment Colab saat notebook dijalankan.

In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests"], cwd=str(REPO_DIR),
        env=LOCAL_ENV, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=240
    )
    print(result.stdout[-6000:])
    if result.returncode != 0:
        raise RuntimeError("Tes repository gagal: " + str(result.returncode))
else:
    print("Tes dilewati")

## 8. Tes quota institusional lokal

Pemeriksaan ini menggunakan SQLite untuk contoh. Pengujian PostgreSQL lintas worker sudah menjadi bagian CI repository, tetapi tidak dijalankan otomatis oleh notebook.

In [ ]:
from im19_mcp.distributed_quota import DistributedQuota
engine = create_engine(LOCAL_SETTINGS["IM19_DATABASE_URL"])
limit = DistributedQuota(engine, secret=LOCAL_SETTINGS["IM19_AUDIT_SALT"])
results = [limit.consume("colab:demo", "test-only", limit=2)[0] for _ in range(3)]
assert results == [True, True, False], results
print("Quota:", results)
engine.dispose()

## 9. Staging HTTP pada localhost (opsional, NONAKTIF)

Mode default Colab tidak bisa langsung dihubungkan ke ChatGPT: belum ada HTTPS publik, issuer OAuth independen, dan callback Google yang disiapkan. Jangan membuat tunnel publik dari notebook.

Jika sudah memiliki PostgreSQL staging dan identity server MCP, simpan credential menggunakan fitur Secrets Colab; gunakan konfigurasi deployment di docs/DEPLOYMENT.md untuk menjalankan server pada lingkungan hosting yang persisten. Repo mendukung im19-gateway setelah konfigurasi tersebut tersedia.

Notebook ini sengaja tidak menghidupkan gateway publik atau menjalankan Google OAuth dengan identitas sintetis.

In [ ]:
ENABLE_PUBLIC_GATEWAY = False
assert ENABLE_PUBLIC_GATEWAY is False, "Jangan expose gateway Colab secara publik."
print("Gateway publik dinonaktifkan. Runtime lokal MCP stdio berjalan lewat sel-sel di atas.")

## 10. Ringkasan

Colab berhasil menjalankan repository jika semua sel berakhir tanpa exception dan smoke_test melaporkan tool registry serta status Drive AUTH_REQUIRED. Ini membuktikan runtime lokal, **bukan** koneksi Google/ChatGPT live atau production readiness.

Jika Colab terputus, ulangi eksekusi sel. Data di /content dan kunci sementara tidak dijamin bertahan. Jangan menyimpan token produksi atau Fernet key bersama database di Google Drive.

In [ ]:
print("COMMIT:", PINNED_SHA)
print("MCP runtime:", MCP_RESULT["transport"])
print("Tools:", len(MCP_RESULT["tools"]))
print("Drive OAuth:", MCP_RESULT["drive_status"])
print("Production ready:", False)
print("ChatGPT remote connected:", False)